In [1]:
# ===========================================================================
# CELL 1: IMPORTS & CONFIG
# ===========================================================================
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "/kaggle/input/competitions/datathon-2026-round-1"
OUTPUT_PATH = "submission.csv"

In [2]:
# ===========================================================================
# CELL 2: LOAD DATA
# ===========================================================================
def load_data():
    print("=" * 60)
    print("STEP 1: Loading data...")
    print("=" * 60)

    sales = pd.read_csv(f"{DATA_DIR}/sales.csv", parse_dates=["Date"])
    sales.columns = ["Date", "Revenue", "COGS"]
    sales = sales.sort_values("Date").reset_index(drop=True)
    print(f"  sales: {sales.shape}, {sales['Date'].min()} -> {sales['Date'].max()}")

    sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv", parse_dates=["Date"])
    print(f"  test dates: {sample_sub.shape}, {sample_sub['Date'].min()} -> {sample_sub['Date'].max()}")

    promotions = pd.read_csv(f"{DATA_DIR}/promotions.csv",
                              parse_dates=["start_date", "end_date"])
    print(f"  promotions: {promotions.shape}")

    web_traffic = pd.read_csv(f"{DATA_DIR}/web_traffic.csv", parse_dates=["date"])
    print(f"  web_traffic: {web_traffic.shape}")

    inventory = pd.read_csv(f"{DATA_DIR}/inventory.csv", parse_dates=["snapshot_date"])
    print(f"  inventory: {inventory.shape}")

    orders = pd.read_csv(f"{DATA_DIR}/orders.csv", parse_dates=["order_date"])
    print(f"  orders: {orders.shape}")

    try:
        customers = pd.read_csv(f"{DATA_DIR}/customers.csv", parse_dates=["signup_date"])
        print(f"  customers: {customers.shape}")
    except:
        customers = None

    return {
        "sales": sales,
        "test": sample_sub,
        "promotions": promotions,
        "web_traffic": web_traffic,
        "inventory": inventory,
        "orders": orders,
        "customers": customers,
    }

data = load_data()

STEP 1: Loading data...
  sales: (3833, 3), 2012-07-04 00:00:00 -> 2022-12-31 00:00:00
  test dates: (548, 3), 2023-01-01 00:00:00 -> 2024-07-01 00:00:00
  promotions: (50, 10)
  web_traffic: (3652, 7)
  inventory: (60247, 17)
  orders: (646945, 8)
  customers: (121930, 7)


In [3]:
# ===========================================================================
# CELL 3: BUSINESS LAYER
# conversion_rate = 1 - bounce_rate (tự tính vì không có trong bảng)
# ===========================================================================
def build_business_signals(data):
    print("\n" + "=" * 60)
    print("STEP 2: Building Business Layer signals...")
    print("=" * 60)

    promotions  = data["promotions"]
    web_traffic = data["web_traffic"]
    inventory   = data["inventory"]
    orders      = data["orders"]

    # 2.1 Promo Schedule
    print("\n  [2.1] Analyzing promotion schedule...")
    print(promotions[["promo_type", "start_date", "end_date",
                       "discount_value", "applicable_category"]].to_string())

    promo_schedule = []
    for _, row in promotions.iterrows():
        promo_schedule.append({
            "name":        row.get("promo_type", "unknown"),
            "start_month": row["start_date"].month,
            "start_day":   row["start_date"].day,
            "duration":    (row["end_date"] - row["start_date"]).days,
            "discount":    row.get("discount_value", 0),
            "category":    row.get("applicable_category", "all"),
        })

    # 2.2 Web Traffic – tự tính conversion_rate
    print("\n  [2.2] Analyzing web traffic patterns...")
    wt = web_traffic.copy()
    wt["month"] = wt["date"].dt.month
    wt["year"]  = wt["date"].dt.year

    # ==============================
    # FIX: conversion_rate = 1 - bounce_rate
    # ==============================
    if "conversion_rate" not in wt.columns:
        print("  -> conversion_rate không có trong file, tính proxy = 1 - bounce_rate")
        wt["sessions"] = wt["sessions"].replace(0, np.nan)
        wt["conversion_rate"] = 1 - wt["bounce_rate"]
    else:
        print("  -> conversion_rate đã có sẵn trong file")

    monthly_conversion = wt.groupby("month").agg(
        avg_sessions=("sessions", "mean"),
        avg_bounce_rate=("bounce_rate", "mean"),
        avg_conversion_rate=("conversion_rate", "mean"),
    ).reset_index()
    print(monthly_conversion.to_string())

    # 2.3 Inventory Health
    print("\n  [2.3] Analyzing inventory health by month...")
    inv = inventory.copy()
    inv["month"] = inv["snapshot_date"].dt.month
    stockout_by_month = inv.groupby("month").agg(
        stockout_rate=("stockout_flag", "mean"),
        overstock_rate=("overstock_flag", "mean"),
        avg_stock=("stock_on_hand", "mean"),
    ).reset_index()
    print(stockout_by_month.to_string())

    # 2.4 Order Source Mix
    print("\n  [2.4] Analyzing order source by month...")
    ord_df = orders.copy()
    ord_df["month"] = ord_df["order_date"].dt.month
    if "order_source" in ord_df.columns:
        source_mix = ord_df.groupby(["month", "order_source"]).size().unstack(fill_value=0)
        source_mix_pct = source_mix.div(source_mix.sum(axis=1), axis=0)
        print(source_mix_pct.to_string())
    else:
        source_mix_pct = None

    # 2.5 Daily Order Volume Pattern
    print("\n  [2.5] Analyzing daily order volume pattern...")
    ord_df["date"] = ord_df["order_date"].dt.date
    daily_orders = ord_df.groupby("date").size().reset_index(name="order_count")
    daily_orders["date"] = pd.to_datetime(daily_orders["date"])
    daily_orders["month"] = daily_orders["date"].dt.month
    monthly_order_pattern = daily_orders.groupby("month")["order_count"].mean().reset_index()
    print(monthly_order_pattern.to_string())

    return {
        "promo_schedule":        promo_schedule,
        "monthly_conversion":    monthly_conversion,
        "stockout_by_month":     stockout_by_month,
        "source_mix_pct":        source_mix_pct,
        "monthly_order_pattern": monthly_order_pattern,
    }

business_signals = build_business_signals(data)


STEP 2: Building Business Layer signals...

  [2.1] Analyzing promotion schedule...
    promo_type start_date   end_date  discount_value applicable_category
0   percentage 2013-03-18 2013-04-17            12.0                 NaN
1   percentage 2013-06-23 2013-07-22            18.0                 NaN
2   percentage 2013-08-30 2013-10-02            10.0                 NaN
3   percentage 2013-11-18 2014-01-02            20.0                 NaN
4        fixed 2013-07-30 2013-09-02            50.0          Streetwear
5   percentage 2013-01-31 2013-03-01            15.0             Outdoor
6   percentage 2014-03-18 2014-04-17            12.0                 NaN
7   percentage 2014-06-23 2014-07-22            18.0                 NaN
8   percentage 2014-08-30 2014-10-01            10.0                 NaN
9   percentage 2014-11-19 2015-01-02            20.0                 NaN
10  percentage 2015-03-18 2015-04-17            12.0                 NaN
11  percentage 2015-06-23 2015-07-22   

In [4]:
# ===========================================================================
# CELL 4: TẾT DATES
# ===========================================================================
TET_DATES = {
    2012: "2012-01-23", 2013: "2013-02-10", 2014: "2014-01-31",
    2015: "2015-02-19", 2016: "2016-02-08", 2017: "2017-01-28",
    2018: "2018-02-16", 2019: "2019-02-05", 2020: "2020-01-25",
    2021: "2021-02-12", 2022: "2022-02-01", 2023: "2023-01-22",
    2024: "2024-02-10",
}
TET_TS = {y: pd.Timestamp(v) for y, v in TET_DATES.items()}

def nearest_tet_diff(dt):
    cands = []
    for offset in [-1, 0, 1]:
        yr = dt.year + offset
        if yr in TET_TS:
            diff = (dt - TET_TS[yr]).days
            if abs(diff) <= 60:
                cands.append(diff)
    if not cands:
        return 999
    return min(cands, key=abs)

print("TET_TS loaded:", len(TET_TS), "years")

TET_TS loaded: 13 years


In [5]:
# ===========================================================================
# CELL 5: PROMO SCHEDULE DEFAULT + VN HOLIDAYS
# ===========================================================================
PROMO_SCHEDULE_DEFAULT = [
    ("spring_sale",    3, 18, 30, 12,   True),
    ("mid_year",       6, 23, 29, 18,   True),
    ("fall_launch",    8, 30, 32, 10,   True),
    ("year_end",      11, 18, 45, 20,   True),
    ("urban_blowout",  7, 30, 33, None, "odd"),
    ("rural_special",  1, 30, 30, 15,   "odd"),
]

VN_FIXED_HOLIDAYS = [
    (1,  1,  "new_year"),
    (2, 14, "valentine"),       
    (3,  8,  "womens_day"),
    (4, 30,  "reunification"),
    (5,  1,  "labor_day"),
    (9,  2,  "national_day"),
    (10, 20, "vn_womens_day"),
    (11, 11, "dd_1111"),
    (12, 12, "dd_1212"),
    (12, 24, "christmas_eve"),
    (12, 25, "christmas"),
]
print("Config loaded.")

Config loaded.


In [6]:
# ===========================================================================
# CELL 6: BUILD FEATURES
# ===========================================================================
def build_features(dates, business_signals=None, promo_schedule=None):
    if promo_schedule is None:
        promo_schedule = PROMO_SCHEDULE_DEFAULT

    df = pd.DataFrame({"Date": dates})
    d  = df["Date"]

    # (A) Calendar
    df["year"]          = d.dt.year
    df["month"]         = d.dt.month
    df["day"]           = d.dt.day
    df["dow"]           = d.dt.dayofweek
    df["doy"]           = d.dt.dayofyear
    df["quarter"]       = d.dt.quarter
    df["is_weekend"]    = (df["dow"] >= 5).astype(int)
    df["is_weekday"]    = (df["dow"] <  5).astype(int)
    df["days_in_month"] = d.dt.days_in_month
    df["days_to_eom"]   = df["days_in_month"] - df["day"]
    df["days_from_som"] = df["day"] - 1
    df["week_of_year"]  = d.dt.isocalendar().week.astype(int)

    # (B) Edge-of-Month
    for k in [1, 2, 3]:
        df[f"is_last{k}"]  = (df["days_to_eom"]  <= k - 1).astype(int)
        df[f"is_first{k}"] = (df["days_from_som"] <= k - 1).astype(int)
    df["is_last_5d"]   = (df["days_to_eom"]  <= 4).astype(int)
    df["is_first_5d"]  = (df["days_from_som"] <= 4).astype(int)
    df["is_last_10d"]  = (df["days_to_eom"]  <= 9).astype(int)
    df["is_mid_month"] = ((df["day"] >= 10) & (df["day"] <= 20)).astype(int)

    # (C) Trend & Regime
    anchor = pd.Timestamp("2020-01-01")
    df["t_days"]          = (d - anchor).dt.days
    df["t_years"]         = df["t_days"] / 365.25
    df["regime_pre2019"]  = (df["year"] <= 2018).astype(int)
    df["regime_2019"]     = (df["year"] == 2019).astype(int)
    df["regime_post2019"] = (df["year"] >= 2020).astype(int)
    df["trend_post2019"]  = np.where(df["year"] >= 2020, df["t_days"], 0)
    df["trend_pre2019"]   = np.where(df["year"] <  2019, df["t_days"], 0)

    # (D) Fourier
    TAU = 2 * np.pi
    for k in range(1, 6):
        df[f"sin_y{k}"] = np.sin(TAU * k * df["doy"] / 365.25)
        df[f"cos_y{k}"] = np.cos(TAU * k * df["doy"] / 365.25)
    for k in range(1, 3):
        df[f"sin_w{k}"] = np.sin(TAU * k * df["dow"] / 7.0)
        df[f"cos_w{k}"] = np.cos(TAU * k * df["dow"] / 7.0)
    for k in range(1, 3):
        df[f"sin_m{k}"] = np.sin(TAU * k * (df["day"] - 1) / df["days_in_month"])
        df[f"cos_m{k}"] = np.cos(TAU * k * (df["day"] - 1) / df["days_in_month"])

    # (E) Tết
    diffs = np.array([nearest_tet_diff(dt) for dt in d])
    df["tet_days_diff"] = diffs
    df["tet_in_3"]      = (np.abs(diffs) <= 3).astype(int)
    df["tet_in_7"]      = (np.abs(diffs) <= 7).astype(int)
    df["tet_in_14"]     = (np.abs(diffs) <= 14).astype(int)
    df["tet_before_7"]  = ((diffs >= -7) & (diffs <  0)).astype(int)
    df["tet_after_7"]   = ((diffs >   0) & (diffs <= 7)).astype(int)
    df["tet_after_14"]  = ((diffs >   7) & (diffs <= 21)).astype(int)
    df["tet_on"]        = (diffs == 0).astype(int)
    df["tet_proximity"] = np.exp(-0.5 * (diffs / 10) ** 2)

    # (F) Holidays
    for (m, dd, name) in VN_FIXED_HOLIDAYS:
        df[f"hol_{name}"] = ((df["month"] == m) & (df["day"] == dd)).astype(int)

    def is_black_friday(dt):
        if dt.month != 11: return 0
        last = pd.Timestamp(year=dt.year, month=11, day=30)
        last_fri = last - pd.Timedelta(days=(last.dayofweek - 4) % 7)
        return int(dt == last_fri)
    df["hol_black_friday"] = [is_black_friday(dt) for dt in d]
    df["hol_double_day"] = (
        ((df["month"] == 11) & (df["day"] == 11)) |
        ((df["month"] == 12) & (df["day"] == 12))
    ).astype(int)

    # (G) Odd/Even Year
    # (G) Odd/Even Year — thêm interactions mạnh
    df["is_odd_year"]  = (df["year"] % 2).astype(int)
    df["is_even_year"] = 1 - df["is_odd_year"]
    df["is_q1"] = (df["quarter"] == 1).astype(int)
    df["is_q2"] = (df["quarter"] == 2).astype(int)
    df["is_q3"] = (df["quarter"] == 3).astype(int)
    df["is_q4"] = (df["quarter"] == 4).astype(int)
    df["q3_odd_year"]  = df["is_q3"] * df["is_odd_year"]
    df["q3_even_year"] = df["is_q3"] * df["is_even_year"]
    df["aug_even"] = ((df["month"] == 8) & df["is_even_year"].astype(bool)).astype(int)
    df["aug_odd"]  = ((df["month"] == 8) & df["is_odd_year"].astype(bool)).astype(int)
    df["jan_odd"]  = ((df["month"] == 1) & df["is_odd_year"].astype(bool)).astype(int)

    # THÊM MỚI: interactions tháng × odd/even year cho các tháng quan trọng
    df["mar_odd"]  = ((df["month"] == 3) & df["is_odd_year"].astype(bool)).astype(int)
    df["mar_even"] = ((df["month"] == 3) & df["is_even_year"].astype(bool)).astype(int)
    df["jun_odd"]  = ((df["month"] == 6) & df["is_odd_year"].astype(bool)).astype(int)
    df["jun_even"] = ((df["month"] == 6) & df["is_even_year"].astype(bool)).astype(int)
    df["nov_odd"]  = ((df["month"] == 11) & df["is_odd_year"].astype(bool)).astype(int)
    df["nov_even"] = ((df["month"] == 11) & df["is_even_year"].astype(bool)).astype(int)
    df["dec_odd"]  = ((df["month"] == 12) & df["is_odd_year"].astype(bool)).astype(int)
    df["dec_even"] = ((df["month"] == 12) & df["is_even_year"].astype(bool)).astype(int)

    # THÊM MỚI: EOM × tháng cao điểm
    df["eom_mar"]  = df["is_last_5d"] * (df["month"] == 3).astype(int)
    df["eom_jun"]  = df["is_last_5d"] * (df["month"] == 6).astype(int)
    df["eom_aug"]  = df["is_last_5d"] * (df["month"] == 8).astype(int)
    df["eom_dec"]  = df["is_last_5d"] * (df["month"] == 12).astype(int)

    # THÊM MỚI: quarter × odd/even year
    df["q1_odd"]  = df["is_q1"] * df["is_odd_year"]
    df["q2_odd"]  = df["is_q2"] * df["is_odd_year"]
    df["q4_odd"]  = df["is_q4"] * df["is_odd_year"]
    df["q1_even"] = df["is_q1"] * df["is_even_year"]
    df["q2_even"] = df["is_q2"] * df["is_even_year"]
    df["q4_even"] = df["is_q4"] * df["is_even_year"]

    # (H) Promo Windows
    all_years = list(range(df["year"].min() - 1, df["year"].max() + 2))
    for (name, sm, sd, dur, disc, recur) in promo_schedule:
        in_promo  = np.zeros(len(df), dtype=int)
        since_arr = np.full(len(df), -1.0)
        until_arr = np.full(len(df), -1.0)
        disc_arr  = np.zeros(len(df))
        for y in all_years:
            if recur == "odd"  and y % 2 == 0: continue
            if recur == "even" and y % 2 != 0: continue
            try:
                start = pd.Timestamp(year=y, month=sm, day=sd)
                end   = start + pd.Timedelta(days=dur)
                mask  = (d >= start) & (d <= end)
                in_promo[mask]  = 1
                since_arr[mask] = (d[mask] - start).dt.days
                until_arr[mask] = (end - d[mask]).dt.days
                disc_arr[mask]  = disc or 0
            except:
                pass
        df[f"promo_{name}"]       = in_promo
        df[f"promo_{name}_since"] = since_arr
        df[f"promo_{name}_until"] = until_arr
        df[f"promo_{name}_disc"]  = disc_arr

    promo_cols = [c for c in df.columns if c.startswith("promo_") and
                  not any(x in c for x in ["since", "until", "disc"])]
    df["total_active_promos"] = df[promo_cols].sum(axis=1)
    df["has_any_promo"]       = (df["total_active_promos"] > 0).astype(int)

    # (I) Business Layer Signals
    # (I) Business Layer Signals — chỉ dùng signal có variation thực
    if business_signals is not None:

        # BỎ traffic_conversion_signal (range ~0.0002, gần flat → noise)
        # BỎ stockout_pressure (range ~0.025, gần flat → noise)

        # CHỈ GIỮ order_volume_signal — range 95→247, variation rõ
        # if business_signals.get("monthly_order_pattern") is not None:
        #     ord_sig = business_signals["monthly_order_pattern"].set_index("month")
        #     mean_orders = ord_sig["order_count"].mean()
        #     df["order_volume_signal"] = df["month"].map(
        #         (ord_sig["order_count"] / mean_orders)
        #     ).fillna(1.0)
        #     print(f"  -> order_volume_signal range: "
        #           f"{df['order_volume_signal'].min():.3f} ~ "
        #           f"{df['order_volume_signal'].max():.3f}")
        pass 

    # (J) Month/Quarter/DOW dummies
    for m in range(1, 13):
        df[f"month_{m}"] = (df["month"] == m).astype(int)
    for q in range(1, 5):
        df[f"quarter_{q}"] = (df["quarter"] == q).astype(int)
    for dow in range(7):
        df[f"dow_{dow}"] = (df["dow"] == dow).astype(int)

    # (K) Season Blocks
    df["is_tet_season"]    = df["month"].isin([1, 2]).astype(int)
    df["is_spring_peak"]   = df["month"].isin([3, 4]).astype(int)
    df["is_summer_low"]    = df["month"].isin([7, 8, 9]).astype(int)
    df["is_year_end_peak"] = df["month"].isin([11, 12]).astype(int)

    df = df.drop(columns=["Date"])
    return df

In [7]:
def compute_sample_weights(dates, scheme="high_era"):
    years = dates.dt.year.values
    n = len(dates)

    if scheme == "high_era":
        w = np.full(n, 0.01)
        w[(years >= 2014) & (years <= 2018)] = 1.0

    elif scheme == "blend_recent":
        w = np.full(n, 0.01)
        w[(years >= 2014) & (years <= 2018)] = 1.0
        w[(years >= 2020) & (years <= 2022)] = 0.8  # tăng từ 0.5 lên 0.8

    elif scheme == "post2019_focus":
        # test là 2023-2024 → ưu tiên học từ 2020-2022
        w = np.full(n, 0.01)
        w[(years >= 2014) & (years <= 2018)] = 0.5
        w[(years >= 2020) & (years <= 2022)] = 1.0

    elif scheme == "uniform":
        w = np.ones(n)
    else:
        w = np.ones(n)

    return w

In [8]:
# ===========================================================================
# CELL 8: MODEL – RIDGE (M1)
# ===========================================================================
def train_ridge(X_train, y_train, alpha=3.0):
    from sklearn.linear_model import Ridge
    mu    = X_train.mean(axis=0)
    sigma = X_train.std(axis=0)
    sigma[sigma == 0] = 1.0
    Xs    = (X_train - mu) / sigma
    model = Ridge(alpha=alpha, random_state=42, max_iter=5000)
    model.fit(Xs, y_train)
    return model, (mu, sigma)

def predict_ridge(model, X_test, stats):
    mu, sigma = stats
    return model.predict((X_test - mu) / sigma)

In [9]:
# ===========================================================================
# CELL 9: MODEL – LIGHTGBM (M2 + Q-Specialists)
# ===========================================================================
def train_lgb(X_train, y_train, sample_weight=None,
              params=None, n_rounds_override=None):
    import lightgbm as lgb

    DEFAULT_PARAMS = dict(
        objective="regression", metric="mae",
        learning_rate=0.03, num_leaves=63,
        min_data_in_leaf=30, feature_fraction=0.85,
        bagging_fraction=0.85, bagging_freq=5,
        lambda_l2=1.0, lambda_l1=0.1,
        seed=42, verbosity=-1, n_jobs=-1,
    )
    if params:
        DEFAULT_PARAMS.update(params)

    n     = len(X_train)
    n_val = min(180, int(n * 0.1))

    if n_rounds_override is None:
        fit_mask = np.ones(n, dtype=bool)
        fit_mask[-n_val:] = False
        val_mask = ~fit_mask

        dtrain = lgb.Dataset(X_train[fit_mask], y_train[fit_mask],
                             weight=sample_weight[fit_mask] if sample_weight is not None else None)
        dval   = lgb.Dataset(X_train[val_mask], y_train[val_mask],
                             weight=sample_weight[val_mask] if sample_weight is not None else None)
        booster_es = lgb.train(
            DEFAULT_PARAMS, dtrain, num_boost_round=5000,
            valid_sets=[dval],
            callbacks=[lgb.early_stopping(300, verbose=False),
                       lgb.log_evaluation(0)]
        )
        best_iter = booster_es.best_iteration
    else:
        best_iter = n_rounds_override

    dtrain_full = lgb.Dataset(X_train, y_train,
                               weight=sample_weight if sample_weight is not None else None)
    booster_final = lgb.train(
        DEFAULT_PARAMS, dtrain_full,
        num_boost_round=best_iter,
        callbacks=[lgb.log_evaluation(0)]
    )
    return {"model": booster_final, "best_iter": best_iter}

def predict_lgb(lgb_result, X_test):
    return lgb_result["model"].predict(X_test)

In [10]:
# ===========================================================================
# CELL 10: MODEL – PROPHET (M3)
# ===========================================================================
def train_prophet(train_df, post_regime_only=True):
    try:
        from prophet import Prophet
    except ImportError:
        try:
            from fbprophet import Prophet
        except ImportError:
            print("  WARNING: Prophet not installed, skipping M3")
            return None

    prophet_df = pd.DataFrame({
        "ds": train_df["Date"],
        "y":  np.log(train_df["Revenue"].clip(lower=1)),
    })
    if post_regime_only:
        prophet_df = prophet_df[prophet_df["ds"] >= "2020-01-01"]

    promo_cols = [c for c in train_df.columns if c.startswith("promo_") and
                  not any(x in c for x in ["since", "until", "disc"])]
    for col in promo_cols:
        prophet_df[col] = train_df.loc[prophet_df.index, col].values if post_regime_only \
                          else train_df[col].values

    m = Prophet(
        yearly_seasonality=True, weekly_seasonality=True,
        daily_seasonality=False, seasonality_mode="multiplicative",
        changepoint_prior_scale=0.05,
    )
    for col in promo_cols:
        m.add_regressor(col)

    m.fit(prophet_df)
    return {"model": m, "promo_cols": promo_cols}

def predict_prophet(prophet_result, test_df):
    if prophet_result is None:
        return None
    m = prophet_result["model"]
    promo_cols = prophet_result["promo_cols"]
    future = pd.DataFrame({"ds": test_df["Date"]})
    for col in promo_cols:
        future[col] = test_df[col].values if col in test_df.columns else 0
    forecast = m.predict(future)
    return np.exp(forecast["yhat"].values)

In [11]:
# ===========================================================================
# CELL 11: CROSS-VALIDATION
# ===========================================================================
def time_series_cv(feat_train, sales, feature_cols, target="Revenue"):
    from sklearn.metrics import mean_absolute_error

    results = {}
    folds = [
        ("A_primary",   "2021-12-31", "2022-01-01", "2022-12-31"),
        ("B_stability", "2020-12-31", "2021-01-01", "2021-12-31"),
        ("C_horizon",   "2021-06-30", "2021-07-01", "2022-06-30"),
    ]

    for fold_name, train_end, val_start, val_end in folds:
        train_mask = sales["Date"] <= train_end
        val_mask   = (sales["Date"] >= val_start) & (sales["Date"] <= val_end)

        X_tr = feat_train.loc[train_mask, feature_cols].values
        y_tr = np.log(sales.loc[train_mask, target].clip(lower=1).values)
        X_vl = feat_train.loc[val_mask, feature_cols].values
        y_vl = sales.loc[val_mask, target].values

        w_tr = compute_sample_weights(sales.loc[train_mask, "Date"], scheme="high_era")
        lgb_result = train_lgb(X_tr, y_tr, sample_weight=w_tr)
        pred = np.exp(predict_lgb(lgb_result, X_vl))
        mae  = mean_absolute_error(y_vl, pred)

        results[fold_name] = {"mae": mae, "n_val": val_mask.sum()}
        print(f"  Fold {fold_name}: MAE = {mae:,.0f} (n={val_mask.sum()})")

    return results

In [12]:
# ===========================================================================
# CELL 12: ENSEMBLE (3-Tier)
# ALPHA=0.60, w_ridge=0.10, w_prophet=0.10, CR=1.26, CC=1.32
# ===========================================================================
def ensemble_predictions(p_ridge, p_lgb_base, p_lgb_spec_composed,
                          p_prophet, alpha=0.9, w_ridge=0.10,
                          w_prophet=0.10, cr=1.26):
    # Tier 1: blend base + Q-specialist
    lgb_blend = (1 - alpha) * p_lgb_base + alpha * p_lgb_spec_composed

    # Tier 2: weighted average
    w_lgb = 1.0 - w_ridge - w_prophet
    if p_prophet is not None:
        raw = w_ridge * p_ridge + w_prophet * p_prophet + w_lgb * lgb_blend
    else:
        raw = w_ridge * p_ridge + (w_lgb + w_prophet) * lgb_blend

    # Tier 3: calibration
    final = cr * raw
    return final

In [13]:
# ===========================================================================
# CELL 13: MAIN PIPELINE
# ===========================================================================
def run_pipeline():
    print("\n" + "=" * 70)
    print("VinDatathon 2026 - Task 3: Sales Forecasting Pipeline")
    print("=" * 70)

    # 9.1 Load
    data       = load_data()
    sales      = data["sales"]
    test       = data["test"]

    # 9.2 Business Layer
    business_signals = build_business_signals(data)

    # 9.3 Features
    print("\n" + "=" * 60)
    print("STEP 3: Building features...")
    print("=" * 60)
    all_dates = pd.concat([pd.Series(sales["Date"]),
                            pd.Series(test["Date"])]).reset_index(drop=True)
    feat_all = build_features(
        pd.DatetimeIndex(all_dates),
        business_signals=business_signals,
        promo_schedule=PROMO_SCHEDULE_DEFAULT,
    )
    feat_all["Date"] = all_dates.values
    n_train    = len(sales)
    feat_train = feat_all.iloc[:n_train].reset_index(drop=True)
    feat_test  = feat_all.iloc[n_train:].reset_index(drop=True)
    feature_cols = [c for c in feat_all.columns if c != "Date"]
    print(f"  Total features: {len(feature_cols)}")

    # 9.4 CV
    print("\n" + "=" * 60)
    print("STEP 4: Cross-validation...")
    print("=" * 60)
    cv_results = time_series_cv(feat_train, sales, feature_cols, target="Revenue")

    # 9.5 Arrays
    X_train = feat_train[feature_cols].values.astype(np.float32)
    X_test  = feat_test[feature_cols].values.astype(np.float32)
    y_train_rev  = np.log(sales["Revenue"].clip(lower=1).values)
    y_train_cogs = np.log(sales["COGS"].clip(lower=1).values)
    w_high_era   = compute_sample_weights(sales["Date"], scheme="high_era")

    # 9.6 Ridge
    print("\n" + "=" * 60)
    print("STEP 5: Training Ridge...")
    print("=" * 60)
    ridge_rev,  stats_rev  = train_ridge(X_train, y_train_rev,  alpha=3.0)
    ridge_cogs, stats_cogs = train_ridge(X_train, y_train_cogs, alpha=3.0)
    p_ridge_rev  = np.exp(predict_ridge(ridge_rev,  X_test, stats_rev))
    p_ridge_cogs = np.exp(predict_ridge(ridge_cogs, X_test, stats_cogs))
    print(f"  Ridge Revenue: mean={p_ridge_rev.mean():.0f}")

    # 9.7 LGB Base
    print("\n" + "=" * 60)
    print("STEP 6: Training LightGBM base...")
    print("=" * 60)
    lgb_rev  = train_lgb(X_train, y_train_rev,  sample_weight=w_high_era)
    lgb_cogs = train_lgb(X_train, y_train_cogs, sample_weight=w_high_era)
    p_lgb_rev  = np.exp(predict_lgb(lgb_rev,  X_test))
    p_lgb_cogs = np.exp(predict_lgb(lgb_cogs, X_test))
    print(f"  LGB Revenue: mean={p_lgb_rev.mean():.0f}")

    # 9.8 Q-Specialists
    print("\n" + "=" * 60)
    print("STEP 7: Training Q-Specialist LGB (Q1-Q4)...")
    print("=" * 60)
    Q_BOOST = 3.0
    q_train = feat_train["Date"].apply(lambda d: pd.Timestamp(d).quarter).values
    q_test  = feat_test["Date"].apply(lambda d: pd.Timestamp(d).quarter).values
    spec_rev  = {}
    spec_cogs = {}
    for target_q in [1, 2, 3, 4]:
        print(f"\n  Training Spec_Q{target_q}...")
        w = w_high_era.copy()
        w[q_train == target_q] *= Q_BOOST
        lgb_q_rev  = train_lgb(X_train, y_train_rev,
                                sample_weight=w,
                                n_rounds_override=lgb_rev["best_iter"])
        lgb_q_cogs = train_lgb(X_train, y_train_cogs,
                                sample_weight=w,
                                n_rounds_override=lgb_cogs["best_iter"])
        spec_rev[target_q]  = np.exp(predict_lgb(lgb_q_rev,  X_test))
        spec_cogs[target_q] = np.exp(predict_lgb(lgb_q_cogs, X_test))
        print(f"    Spec_Q{target_q} Rev: mean={spec_rev[target_q].mean():.0f}")

    # Compose Q-specialists
    p_spec_rev  = np.zeros(len(X_test))
    p_spec_cogs = np.zeros(len(X_test))
    for q in [1, 2, 3, 4]:
        mask = q_test == q
        p_spec_rev[mask]  = spec_rev[q][mask]
        p_spec_cogs[mask] = spec_cogs[q][mask]

    # 9.9 Prophet
    print("\n" + "=" * 60)
    print("STEP 8: Training Prophet...")
    print("=" * 60)
    train_df_prophet = pd.concat([sales[["Date", "Revenue"]], feat_train], axis=1)
    train_df_prophet = train_df_prophet.loc[:, ~train_df_prophet.columns.duplicated()]
    prophet_rev = train_prophet(train_df_prophet, post_regime_only=True)

    test_df_prophet = pd.concat([test[["Date"]], feat_test], axis=1)
    test_df_prophet = test_df_prophet.loc[:, ~test_df_prophet.columns.duplicated()]
    p_prophet_rev = predict_prophet(prophet_rev, test_df_prophet)
    if p_prophet_rev is not None:
        print(f"  Prophet Revenue: mean={p_prophet_rev.mean():.0f}")
    else:
        print("  Prophet skipped")

    # 9.10 Ensemble – theo công thức chuẩn
    ALPHA = 0.80  # giữ best
    CR    = 1.26
    CC    = 1.32

    lgb_blend_rev = ALPHA * p_spec_rev  + (1 - ALPHA) * p_lgb_rev
    lgb_blend_cog = ALPHA * p_spec_cogs + (1 - ALPHA) * p_lgb_cogs

    # Bỏ Prophet — dồn weight về LGB
    raw_rev = 0.20 * p_ridge_rev + 0.80 * lgb_blend_rev
    raw_cog = 0.20 * p_ridge_cogs + 0.80 * lgb_blend_cog

    final_rev = CR * raw_rev
    final_cog = CC * raw_cog

    print(f"Final Revenue range: {final_rev.min():.0f} ~ {final_rev.max():.0f}")
    print(f"Final COGS    range: {final_cog.min():.0f} ~ {final_cog.max():.0f}")
    print(f"Mean COGS/Revenue ratio: {(final_cog / final_rev).mean():.4f}")

    # 9.11 Submission
    submission = pd.DataFrame({
        "Date":    test["Date"].dt.strftime("%Y-%m-%d"),
        "Revenue": np.round(final_rev, 2),
        "COGS":    np.round(final_cog, 2),
    })
    submission.to_csv(OUTPUT_PATH, index=False)
    print(f"\n  Submission saved: {OUTPUT_PATH}")
    print(submission.head(10).to_string())

    return submission, cv_results

submission, cv_results = run_pipeline()


VinDatathon 2026 - Task 3: Sales Forecasting Pipeline
STEP 1: Loading data...
  sales: (3833, 3), 2012-07-04 00:00:00 -> 2022-12-31 00:00:00
  test dates: (548, 3), 2023-01-01 00:00:00 -> 2024-07-01 00:00:00
  promotions: (50, 10)
  web_traffic: (3652, 7)
  inventory: (60247, 17)
  orders: (646945, 8)
  customers: (121930, 7)

STEP 2: Building Business Layer signals...

  [2.1] Analyzing promotion schedule...
    promo_type start_date   end_date  discount_value applicable_category
0   percentage 2013-03-18 2013-04-17            12.0                 NaN
1   percentage 2013-06-23 2013-07-22            18.0                 NaN
2   percentage 2013-08-30 2013-10-02            10.0                 NaN
3   percentage 2013-11-18 2014-01-02            20.0                 NaN
4        fixed 2013-07-30 2013-09-02            50.0          Streetwear
5   percentage 2013-01-31 2013-03-01            15.0             Outdoor
6   percentage 2014-03-18 2014-04-17            12.0                 NaN
7 

15:02:42 - cmdstanpy - INFO - Chain [1] start processing
15:02:43 - cmdstanpy - INFO - Chain [1] done processing


  Prophet Revenue: mean=3556533
Final Revenue range: 1100593 ~ 12426988
Final COGS    range: 946220 ~ 10928361
Mean COGS/Revenue ratio: 0.9447

  Submission saved: submission.csv
         Date     Revenue        COGS
0  2023-01-01  2649217.49  2848562.39
1  2023-01-02  1631528.45  1451713.56
2  2023-01-03  1567383.10  1470320.15
3  2023-01-04  1100593.24   946220.16
4  2023-01-05  1317702.46  1213835.04
5  2023-01-06  1659777.66  1475306.95
6  2023-01-07  1568946.31  1432075.63
7  2023-01-08  1797815.94  1679758.26
8  2023-01-09  2245789.77  2120564.55
9  2023-01-10  2010362.67  1878068.82
